In [ ]:
from typing import Optional,List
from dataclasses import dataclass
import os
@dataclass
class TravelState:
  source:Optional[str]=None
  destination:Optional[str]=None
  start_date:Optional[str]=None
  end_date:Optional[str]=None
  budget:Optional[int]=None
  travelers:Optional[int]=None
  preferences:List[str]=None

In [ ]:
from google.colab import userdata
from dataclasses import dataclass

@dataclass
class GeminiConfig:
  api_key:str
  model:str="gemini-2.5-flash"
  temperature:float=0.3
  max_tokens:int=1024

  def __post__init__(self):
    if not self.api_key:
      raise ValueError("API key is required")

def load_gemini_config()->GeminiConfig:
  return GeminiConfig(
      api_key=userdata.get("GOOGLE_API_KEY")
  )

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Configure the generative AI library with your API key
API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=API_KEY)

print("Official Google Generative AI library configured.")

Official Google Generative AI library configured.


In [ ]:
# Initialize a Gemini model and make a simple request
# We'll use 'gemini-pro' as it's a generally available model.
model_official = genai.GenerativeModel('gemini-2.5-flash')

try:
    response_official = model_official.generate_content("Explain AI in one line")
    print("Official library response:")
    print(response_official.text)
except Exception as e:
    print(f"Error using official library: {e}")

Official library response:
AI is machines performing tasks that normally require human intelligence.


In [ ]:
import requests
import json

class LLM:
    def __init__(self, config):
        self.config = config
        self.endpoint = (
            f"https://generativelanguage.googleapis.com/v1/models/"
            f"{config.model}:generateContent"
        )

    def chat(self, prompt: str) -> str:
        payload = {
            "contents": [
                {
                    "parts": [
                        {"text": prompt}
                    ]
                }
            ],
            "generationConfig": {
                "temperature": self.config.temperature,
                "maxOutputTokens": self.config.max_tokens
            }
        }

        response = requests.post(
            f"{self.endpoint}?key={self.config.api_key}",
            headers={"Content-Type": "application/json"},
            data=json.dumps(payload),
            timeout=30
        )

        response.raise_for_status()
        data = response.json()

        return data["candidates"][0]["content"]["parts"][0]["text"]

In [ ]:
config = load_gemini_config() # Use the helper function
llm = LLM(config)

print(llm.chat("Explain AI in one line"))

AI enables machines to perform tasks typically requiring human intelligence.


In [ ]:
import json

class IntentAgent:
    def __init__(self, llm):
        self.llm = llm

    def extract(self, user_input: str) -> dict:
        prompt = f"""
Extract travel details as STRICT JSON.
Rules:
- Use double quotes only
- No comments
- No trailing commas
- No explanations
- Output JSON ONLY

Text:
{user_input}

JSON schema:
{{
  "source": string | null,
  "destination": string | null,
  "start_date": string | null,
  "end_date": string | null,
  "budget": number | null,
  "travelers": number | null,
  "preferences": list[string]
}}
"""

        raw = self.llm.chat(prompt)

        # Strip markdown code block fences if present
        if raw.startswith("```json") and raw.endswith("```"):
            raw = raw[len("```json"): -len("```")].strip()
        elif raw.startswith("```") and raw.endswith("```"):
            raw = raw[len("```"): -len("```")].strip()

        try:
            return json.loads(raw)
        except json.JSONDecodeError as e:
            raise ValueError(
                f"Invalid JSON returned by LLM.\nRaw output:\n{raw}"
            ) from e

In [ ]:
class PlannerAgent:
  def plan(self,state:TravelState):
    steps=[]
    if not state.source or not state.destination:
      steps.append("ask location")
    if not state.start_date:
      steps.append("ask dates")

    steps.extend([
        "search_flights",
        "search_hotels",
        "search_activites",
        "build_itinerary"
    ])
    return steps

In [ ]:
from typing import List, Dict

class FlightTool:
    def search(self, source: str, destination: str, date: str, travelers: int) -> List[Dict]:
        raise NotImplementedError


class HotelTool:
    def search(self, destination: str, nights: int, budget: int) -> List[Dict]:
        raise NotImplementedError


class ActivityTool:
    def search(self, destination: str, preferences: list[str]) -> List[str]:
        raise NotImplementedError

In [ ]:
import requests
import time
from typing import Optional, List, Dict

from google.colab import userdata


class FlightTool(FlightTool):
  def __init__(self, api_key: Optional[str] = None, api_secret: Optional[str] = None):
    self.api_key = api_key if api_key else userdata.get("AMADEUS_API_KEY")
    self.api_secret = api_secret if api_secret else userdata.get("AMADEUS_API_SECRET")
    if not self.api_key or not self.api_secret:
      raise ValueError("Amadeus API key and secret must be provided or set in userdata.")
    self.base_url="https://test.api.amadeus.com"
    self.token=self._get_token()

  def _get_token(self):
    resp=requests.post(
        f"{self.base_url}/v1/security/oauth2/token",
        data={
            "grant_type":"client_credentials",
            "client_id":self.api_key,
            "client_secret":self.api_secret
        }
    )
    resp.raise_for_status()
    return resp.json()["access_token"]

  def search(self,source:str,destination:str,data:str,travelers:int) -> List[Dict]:
    headers={"Authorization":f"Bearer {self.token}"}
    params={
        "originLocationCode":source,
        "destinationLocationCode":destination,
        "departureDate":data,
        "adults":travelers,
        "currencyCode":"INR",
        "max":5
    }
    resp=requests.get(
        f"{self.base_url}/v2/shopping/flight-offers",
        headers=headers,
        params=params,
        timeout=15
    )
    resp.raise_for_status()
    offers=[]

    for o in resp.json()["data"]:
      offers.append({
          "airline":o["validatingAirlineCodes"][0],
          "price":float(o["price"]["grandTotal"]),
          "stops":len(o["itineraries"][0]["segments"]),
          "duration":o["itineraries"][0]["duration"]
      })
    return offers

In [ ]:
class FlightAgent:
  def search(self,state:TravelState):
    return {
        "airline":"Indigo",
        "price":850,
        "duration":"2h 10m"
    }

In [ ]:
class BookingHotelTool(HotelTool):
  def __init__(self,api_key: Optional[str]=None):
    self.api_key=api_key if api_key else userdata.get("RAPIDAPI_KEY") # Assuming a key name
    if not self.api_key:
      raise ValueError("RapidAPI key must be provided or set in userdata.")
    self.base_url="https://booking-com.p.rapidapi.com"

  def search(self,destination: str,nights: int,budget: int) -> List[Dict]:
    headers={
        "X-RapidAPI-Key":self.api_key,
        "X-RapidAPI-Host":"booking-com15.p.rapidapi.com"
    }
    params={
        "dest_type":"city",
        "dest_id":destination,
        "units":"metric",
        "room_number":1,
        "checkin_date":"2025-01-10",
        "checkout_date":"2025-01-15",
        "adults_number":1,
        "order_by":"price",
        "filter_by_currency":"INR",
    }
    resp=requests.get(
        f"{self.base_url}/v1/hotels/search",
        headers=headers,
        params=params,
        timeout=15
    )
    resp.raise_for_status()
    hotels_data=[]

    for h in resp.json()["results"][:5]:
      hotels_data.append({
          "name":h["name"],
          "price_per_night":h["min_total_price"]/nights,
          "rating":h.get("review_score",0)
      })
    return hotels_data

In [ ]:
class HotelAgent:
  def search(self,state:TravelState):

    return {
        "name":"Taj Residency",
        "price_per_night":4200,
        "rating":4.5,
        "nights": 4
    }

In [ ]:
class WikipediaAcitvityTool(ActivityTool):
  def __init__(self):
    self.base_url="https://en.wikipedia.org/api/rest_v1/page/summary"

  def search(self,destination:str,preferences:list[str])->list[str]:
    url=f"{self.base_url}/{destination}"
    resp=requests.get(url,
                      headers={"User-Agent":"TravelAgent"},
                      timeout=10
                      )


    if resp.status_code!=200:
      return [f"Explore Popular attraction in {destination}"]

    data=resp.json()

    activities=[]
    activities.append(f"Explore {destination} city")
    activities.append(f"Learn about the history of {destination}")
    activities.append(f"See the famous landmarks in {destination}")
    activities.append(f"Experience the culture of {destination}")

    if "description" in data:
      activities.append(data["description"])
    if preferences:
      if any("beach" in p.lower() for p in preferences):
        activities.append("Go to the beach")
      if any("hiking" in p.lower() for p in preferences):
        activities.append("Go for a hike")
      if any("food" in p.lower() for p in preferences):
        activities.append("Try local cuisine")
      if any("nature" in p.lower() for p in preferences):
        activities.append("Explore nearby natural attractions")

    return activities[:5]

In [ ]:
class ActivityAgent:
  def suggest(self,state):
    return [

        "City tour"
        "Local food walk"
    ]

In [ ]:
class ItineraryAgent:
  def build(self,flight,hotel,activities):
    return{
        "flight":flight,
        "hotel":hotel,
        "activities":activities
    }

In [ ]:
class TravelAgent:
    def __init__(self, flight_agent, hotel_agent, activity_agent):
        self.flight_agent = flight_agent
        self.hotel_agent = hotel_agent
        self.activity_agent = activity_agent

    def run(self, state: TravelState) -> dict:
        flight = self.flight_agent.choose(state)
        hotel = self.hotel_agent.choose(state)
        activities = self.activity_agent.choose(state)

        total_cost = (
            flight["price"] * state.travelers +
            hotel["price_per_night"] * 5
        )

        return {
            "trip_summary": state.__dict__,
            "selected_flight": flight,
            "selected_hotel": hotel,
            "activities": activities,
            "cost_breakdown": {
                "estimated_total": total_cost
            },
            "assumptions": [
                "Flights from Amadeus API",
                "Hotels from Booking.com via RapidAPI",
                "Activities from Wikipedia",
                "Availability not guaranteed"
            ]
        }

In [ ]:
state = TravelState(
    source="BOM",
    destination="Goa",
    start_date="2025-01-10",
    end_date="2025-01-15",
    budget=50000,
    travelers=2,
    preferences=["January", "beach"]
)

agent = TravelAgent(
    flight_agent=FlightAgent(FlightTool()),
    hotel_agent=HotelAgent(BookingHotelTool()),
    activity_agent=ActivityAgent(ActivityTool())
)

result = agent.run(state)
print(json.dumps(result, indent=2))

In [ ]:
config = load_gemini_config()
llm = LLM(config)

intent_agent = IntentAgent(llm)
planner = PlannerAgent()
flight_agent = FlightAgent()
hotel_agent = HotelAgent()
activity_agent = ActivityAgent()

agent = TravelAgent(
    intent_agent=intent_agent,
    planner=planner,
    flight_agent=flight_agent,
    hotel_agent=hotel_agent,
    activity_agent=activity_agent
)

result = agent.run(
    "Plan a 5 day Goa trip from Mumbai in January under 50k for 2 people"
)

print(result)